# Jetsound: Sound detection with Jetson Nano

## Data collection and feature extraction

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import os
import cv2
import librosa

DATADIR = 'music_dataset'
CATEGORIES = ['Acoustic_guitar', 'Drum_set', 'vibraphone']

Function to make spectrograms.

In [ ]:
def convert_to_spectrogram(audio_list, sr_list, n_mels, fmax=8000):
    # audio_list : list of librosa audio signals 
    # sr_list : list of sample rates (int)
    # n_mels :
    # fmax : maximum frequency in spectrogram

    # Returns a 4D tensorflow tensor (num_samples, n_mels, time_steps, 1)

    spectrograms = []

    for audio_signal, sr in zip (audio_list, sr_list):
        
        # Create mel spectrogram
        mel_spec = librosa.feature.melspectrogram(
            y = audio_signal,
            sr = sr,
            n_mels = n_mels,
            fmax = fmax
        )
    
        # Convert to decibels
        mel_spec = librosa.power_to_db(mel_spec, ref=np.max)

        # Normalize
        mel_spec = (mel_spec - mel_spec.min()) / (mel_spec.max() - mel_spec.min())

        # Add required dimensions
        mel_spec = mel_spec[..., np.newaxis]  # shape: (n_mels, time_steps, 1)

        spectrograms.append(mel_spec)

    return np.stack(spectrograms, axis=0)

In [ ]:
dataset = []
N_MELS = 128 # Y resolution of the spectrograms

def create_dataset():
    for category in CATEGORIES:
        path = os.path.join(DATADIR, category)
        class_num = CATEGORIES.index(category) #Assigns category 0 to Autistic_guitar, 1 to Drum_set and 2 to vibraphone
        for audio_sample in os.listdir(path):
            try:
                audio, sr = librosa.load(os.path.join(path, audio_sample))
                spectro = convert_to_spectrogram(audio, sr, N_MELS, fmax=8000)
                dataset.append([spectro, class_num]) # Appends the category to the array
            except Exception as e:
                pass
    
create_dataset()

print(len(dataset))

Shuffle the dataset and make sure by looking at the labels.

In [ ]:
import random

random.shuffle(dataset)

for sample in dataset[:10]:
    print(sample[1])

Create your train and test split.

In [ ]:
x = [] #Images
y = [] #Labels

for features, label in dataset:
    x.append(features)  #Place all images in x
    y.append(label)     #Place all labels in y

x = np.array(x).reshape(-1, IMG_SIZE, IMG_SIZE, 1)
y = np.array(y)

# We have to split the data for testing
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=42)

## Network architecture

## Model training

## Validation

You need the library "pyaudio" to test this feature.

    pip install pyaudio
    
    conda install conda-forge::pyaudio

In [3]:
import pyaudio
import wave

# Configuration parameters
FORMAT = pyaudio.paInt16  # 16-bit resolution
CHANNELS = 1              # Mono recording
RATE = 44100              # Sample rate (44.1kHz is standard)
CHUNK = 1024              # Buffer size
RECORD_SECONDS = 5        # Duration of recording
OUTPUT_FILENAME = "test_recording.wav"

audio = pyaudio.PyAudio()

# Start Recording
stream = audio.open(format=FORMAT, channels=CHANNELS,
                    rate=RATE, input=True,
                    frames_per_buffer=CHUNK)

print("Recording...")

frames = []

for i in range(0, int(RATE / CHUNK * RECORD_SECONDS)):
    data = stream.read(CHUNK)
    frames.append(data)

print("Finished recording.")

# Stop and close the stream
stream.stop_stream()
stream.close()
audio.terminate()

# Save the recorded data as a WAV file (THIS NEEDS TO BE CHANGED SO IT IS STORED IN MEMORY)
with wave.open(OUTPUT_FILENAME, 'wb') as wf:
    wf.setnchannels(CHANNELS)
    wf.setsampwidth(audio.get_sample_size(FORMAT))
    wf.setframerate(RATE)
    wf.writeframes(b''.join(frames))

print(f"File saved as {OUTPUT_FILENAME}")

Recording...
Finished recording.
File saved as test_recording.wav
